# 03 - Service Modeling: Travel Time and Service Time

This notebook implements and validates the service models:
- **Travel time**: Based on Haversine/Manhattan distance with time-of-day speed adjustments
- **Service time**: LogNormal distribution fitted to EMS on-scene durations
- **Distance matrices**: Firehouse-to-precinct distance computation

**Runtime:** ~3 minutes  
**Data Required:** Firehouses, distance matrices

In [ ]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
DOWNLOAD_OUTPUTS = False  # Set True to download output files
SAVE_TO_DRIVE = False     # Set True to save outputs to Google Drive

if IN_COLAB:
    print("Running in Google Colab - installing dependencies...")
    !pip install -q simpy pulp pyyaml tqdm
    if not os.path.exists('ems-optimization'):
        !git clone --depth=1 https://github.com/cnsp/ems-optimization.git
    PROJECT_ROOT = '/content/ems-optimization'
else:
    print("Running locally")
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
DATA_DIR = os.path.join(PROJECT_ROOT, 'data')
RAW_DIR = os.path.join(DATA_DIR, 'raw')
PROCESSED_DIR = os.path.join(DATA_DIR, 'processed')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
CONFIGS_DIR = os.path.join(PROJECT_ROOT, 'configs')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

# Optional Google Drive save
if IN_COLAB and SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = '/content/drive/MyDrive/EMS_Optimization_Results'
    os.makedirs(DRIVE_DIR, exist_ok=True)
    print(f"Saving outputs to: {DRIVE_DIR}")

def save_output(fig_or_df, filename, subdir=''):
    """Helper to save outputs with optional download/drive save."""
    out_dir = os.path.join(RESULTS_DIR, subdir) if subdir else RESULTS_DIR
    os.makedirs(out_dir, exist_ok=True)
    filepath = os.path.join(out_dir, filename)
    if isinstance(fig_or_df, pd.DataFrame):
        fig_or_df.to_csv(filepath, index=True)
    elif hasattr(fig_or_df, 'savefig'):
        fig_or_df.savefig(filepath, bbox_inches='tight', dpi=150)
    if IN_COLAB and DOWNLOAD_OUTPUTS:
        from google.colab import files
        files.download(filepath)
    if IN_COLAB and SAVE_TO_DRIVE:
        import shutil
        drive_path = os.path.join(DRIVE_DIR, subdir)
        os.makedirs(drive_path, exist_ok=True)
        shutil.copy(filepath, os.path.join(drive_path, filename))

print("Setup complete. PROJECT_ROOT:", PROJECT_ROOT)

## Load Firehouse and Precinct Data

In [ ]:
firehouses = pd.read_csv(os.path.join(PROCESSED_DIR, 'firehouses_manhattan.csv'))
print(f"Manhattan firehouses: {len(firehouses)}")
print(f"In CBD: {firehouses['in_cbd'].sum()}")
display(firehouses[['FacilityName', 'Latitude', 'Longitude', 'in_cbd']].head(10))

## Distance Matrices

In [ ]:
from ems_readiness.utils.distance import haversine, manhattan_distance, build_distance_matrix

# Load pre-computed distance matrices
dm_haversine = pd.read_csv(os.path.join(PROCESSED_DIR, 'distance_matrix_firehouse_precinct.csv'), index_col=0)
print(f"Haversine distance matrix: {dm_haversine.shape}")
print(f"  Range: {dm_haversine.values.min():.3f} to {dm_haversine.values.max():.3f} miles")
print(f"  Mean: {dm_haversine.values.mean():.3f} miles")

dm_manhattan_path = os.path.join(PROCESSED_DIR, 'distance_matrix_firehouse_precinct_manhattan.csv')
if os.path.exists(dm_manhattan_path):
    dm_manhattan = pd.read_csv(dm_manhattan_path, index_col=0)
    print(f"\nManhattan distance matrix: {dm_manhattan.shape}")
    print(f"  Range: {dm_manhattan.values.min():.3f} to {dm_manhattan.values.max():.3f} miles")
    print(f"  Mean: {dm_manhattan.values.mean():.3f} miles")
    print(f"  Ratio (Manhattan/Haversine mean): {dm_manhattan.values.mean() / dm_haversine.values.mean():.3f}")

### Distance Matrix Heatmap

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

sns.heatmap(dm_haversine, cmap='YlOrRd', ax=axes[0], xticklabels=True, yticklabels=True)
axes[0].set_title('Haversine Distance Matrix (miles)')
axes[0].tick_params(axis='both', labelsize=6)

if os.path.exists(dm_manhattan_path):
    sns.heatmap(dm_manhattan, cmap='YlOrRd', ax=axes[1], xticklabels=True, yticklabels=True)
    axes[1].set_title('Manhattan Distance Matrix (miles)')
    axes[1].tick_params(axis='both', labelsize=6)

plt.tight_layout()
save_output(fig, 'distance_matrices_heatmap.png', 'figures/service')
plt.show()

## Travel Time Model

In [ ]:
from ems_readiness.service.travel_time import (
    travel_time_minutes, build_travel_time_matrix,
    DEFAULT_SPEED_MPH, TOD_SPEED_FACTORS
)

print(f"Default speed: {DEFAULT_SPEED_MPH} mph")
print(f"\nTime-of-Day Speed Factors:")
for hour, factor in sorted(TOD_SPEED_FACTORS.items()):
    effective = DEFAULT_SPEED_MPH * factor
    print(f"  Hour {hour:2d}: factor={factor:.2f}, effective speed={effective:.1f} mph")

# Build travel time matrices for different hours
tt_noon = build_travel_time_matrix(dm_haversine, hour_of_day=12)
tt_rush = build_travel_time_matrix(dm_haversine, hour_of_day=17)
tt_night = build_travel_time_matrix(dm_haversine, hour_of_day=3)

print(f"\nTravel Time Summary (Haversine):")
print(f"  Noon (12:00):  mean={tt_noon.values.mean():.2f} min, max={tt_noon.values.max():.2f} min")
print(f"  Rush (17:00):  mean={tt_rush.values.mean():.2f} min, max={tt_rush.values.max():.2f} min")
print(f"  Night (03:00): mean={tt_night.values.mean():.2f} min, max={tt_night.values.max():.2f} min")

### Travel Time Distribution by Time of Day

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
hours = range(24)
mean_tt = []
for h in hours:
    tt = build_travel_time_matrix(dm_haversine, hour_of_day=h)
    mean_tt.append(tt.values.mean())

ax.plot(hours, mean_tt, 'b-o', markersize=5)
ax.set_xlabel('Hour of Day')
ax.set_ylabel('Mean Travel Time (minutes)')
ax.set_title('Average Travel Time by Time of Day')
ax.axhline(y=np.mean(mean_tt), color='red', linestyle='--', label=f'Overall mean: {np.mean(mean_tt):.2f} min')
ax.legend()
ax.set_xticks(range(0, 24, 2))
plt.tight_layout()
save_output(fig, 'travel_time_by_tod.png', 'figures/service')
plt.show()

## Service Time Model (LogNormal)

In [ ]:
from ems_readiness.service.service_time import ServiceTimeModel

# Default parameters
stm = ServiceTimeModel(mean_minutes=25.0, std_minutes=10.0, distribution='lognormal')
samples = stm.sample(size=10000, rng=42)

print(f"Service Time Distribution (LogNormal)")
print(f"  Target mean: 25.0 min, Target std: 10.0 min")
print(f"  Sample mean: {np.mean(samples):.2f} min")
print(f"  Sample std: {np.std(samples):.2f} min")
print(f"  Sample median: {np.median(samples):.2f} min")
print(f"  Sample p90: {np.percentile(samples, 90):.2f} min")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(samples, bins=60, density=True, color='steelblue', edgecolor='white', alpha=0.7)
axes[0].axvline(x=np.mean(samples), color='red', linestyle='--', label=f'Mean: {np.mean(samples):.1f} min')
axes[0].axvline(x=np.median(samples), color='orange', linestyle='--', label=f'Median: {np.median(samples):.1f} min')
axes[0].set_xlabel('Service Time (minutes)')
axes[0].set_ylabel('Density')
axes[0].set_title('Service Time Distribution (LogNormal)')
axes[0].legend()

# Compare distributions
stm_exp = ServiceTimeModel(mean_minutes=25.0, distribution='exponential')
samples_exp = stm_exp.sample(size=10000, rng=42)
axes[1].hist(samples, bins=60, density=True, alpha=0.5, label='LogNormal', color='steelblue')
axes[1].hist(samples_exp, bins=60, density=True, alpha=0.5, label='Exponential', color='coral')
axes[1].set_xlabel('Service Time (minutes)')
axes[1].set_ylabel('Density')
axes[1].set_title('LogNormal vs Exponential Distribution')
axes[1].legend()

plt.tight_layout()
save_output(fig, 'service_time_distribution.png', 'figures/service')
plt.show()

## Full EMS Call Timeline Example

In [ ]:
print("Example EMS Call Timeline:")
print("=" * 50)

dispatch_delay = 1.5  # minutes (fixed)
travel_time = travel_time_minutes(2.5, hour_of_day=14)  # 2.5 miles at 2pm
service_time = stm.sample(size=1, rng=123)[0]
total_response = dispatch_delay + travel_time
total_call = total_response + service_time

print(f"1. Dispatch delay:     {dispatch_delay:.1f} min")
print(f"2. Travel time:        {travel_time:.1f} min (2.5 miles at 14:00)")
print(f"3. Response time:      {total_response:.1f} min (dispatch + travel)")
print(f"4. Service time:       {service_time:.1f} min")
print(f"5. Total call time:    {total_call:.1f} min")
print(f"\nResponse threshold: 8.0 min")
print(f"Within threshold: {'Yes' if total_response <= 8.0 else 'No'}") 

## Summary

- Haversine distance matrix: 48 firehouses x 30 precincts (mean 4.3 miles)
- Manhattan distance provides conservative upper-bound (~1.2-1.3x Haversine)
- Travel time varies 20-50% based on time-of-day speed factors
- Service time follows LogNormal(mu, sigma) with mean=25 min, std=10 min
- Fixed dispatch delay: 1.5 minutes